# Import packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np
import math
import os

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import PredictionErrorDisplay

from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.pipeline import Pipeline


In [ ]:
# uncomment when using Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Import data

In [ ]:
data_directory = r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Landsat Sampling\Merged Landsat Data'

In [ ]:
df = pd.read_csv(os.path.join(data_directory,'min_date.csv'))
df = df.drop(columns=['Unnamed: 0', 'CHLOROPHYLL',  
                      'CHLOROPHYLL_B', 'DOC', 'dif_date_point',
                      'N_TOTAL', 'N_TOTAL_DISSOLVED', 
                      'POC', 'P_ORGANIC', 'P_TOTAL', 
                      'SILICA',  'TOC', 'duplicated'],axis=1).rename(columns={'SPM':"TSS"})
df.columns

# Evaluation metrics functions

In [ ]:
def model_metrics(y_true,y_pred):
    ''' y = observed target values
    y_pred = predicted target values'''
    from sklearn.metrics import r2_score
    from sklearn.metrics import mean_absolute_error
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_absolute_percentage_error
    from sklearn.metrics import explained_variance_score

    return {'r2':r2_score(y_true, y_pred),
'mae':mean_absolute_error(y_true, y_pred),
'mse':mean_squared_error(y_true, y_pred),
'mape':mean_absolute_percentage_error(y_true, y_pred),
'exp_var': explained_variance_score(y_true, y_pred)
    }

In [ ]:
def cv_model_metrics(model,X,y,n_cv=5):
    ''' model = model to evaluate
    X = predictors
    y = observed target values
    cv = number of cross validations, standard is 5'''
    from sklearn.model_selection import ShuffleSplit

    cv = ShuffleSplit(n_splits=n_cv, test_size=0.15, random_state=0)
    from sklearn.model_selection import cross_val_score

    return {'r2':float(abs(cross_val_score(model, X, y, cv=cv,scoring='r2')).mean()),
'mae':float(abs(cross_val_score(model, X, y, cv=cv,scoring='neg_mean_absolute_error')).mean()),
'mse':float(abs(cross_val_score(model, X, y, cv=cv,scoring='neg_mean_squared_error')).mean()),
'mape':float(abs(cross_val_score(model, X, y, cv=cv,scoring='neg_mean_absolute_percentage_error')).mean()),
'exp_var':float( abs(cross_val_score(model, X, y, cv=cv,scoring='explained_variance')).mean())
    }

# TSS

## Model Regressions

In [ ]:
ols = LinearRegression()

In [ ]:
model_poly = Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)),
                  ('linear', LinearRegression(positive=False))])

In [ ]:
model_splines = Pipeline([('poly', SplineTransformer(n_knots=5, degree=3)),
                  ('linear', LinearRegression(positive=False))])

## subset TSS and satellite data

In [ ]:
df_subset = df[['TSS','blue_mean',
       'green_mean',
       'nir_mean',
       'red_mean',
       'datetime',
       'WATER_PERIOD']].copy()
# retirar valores em branco
df_subset = df_subset.dropna()
df_subset.isna().sum()

In [ ]:
# transform to datetime format
df_subset['date'] = df_subset['datetime'].apply(lambda row: row[:10])
# df_subset.dtypes
df_subset['date']

In [ ]:
drop_columns_X_model = ['TSS','datetime','WATER_PERIOD','date']

### Polynomial NIR regression

In [ ]:
# separate parameters
y = df_subset['TSS'].copy()
X = np.array(df_subset['nir_mean']).reshape(-1, 1).copy()

In [ ]:
model_fit = model_poly.fit(X, y)

In [ ]:
print(f"coefficients: {model_fit.named_steps['linear'].coef_}")
print(f"intercept: {model_fit.named_steps['linear'].intercept_}")

In [ ]:
y_pred_model = model_fit.predict(X)

#### Evaluate results

In [ ]:
metrics = model_metrics(y,y_pred_model)
print('Metrics:')
print(metrics)

Cross validation

In [ ]:
cv_metrics = cv_model_metrics(model=model_poly,X=X,y=y,n_cv=100)
print('CV Metrics:')
print(cv_metrics)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="actual_vs_predicted",
    ax=axs[0]
)
axs[0].set_title("Actual vs. Predicted values")
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="residual_vs_predicted",
    ax=axs[1]
)
axs[1].set_title("Residuals vs. Predicted Values")
fig.suptitle("Plotting predictions")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Create a list of 202 empty strings
total_len = len(y_pred_model)
print(total_len)
ticks = np.arange(0,total_len ,20)
print(ticks)
ticks_len = len(ticks)
my_list = []


for i in range(0, ticks_len):
    my_list.append(df_subset['date'].iloc[ticks[i]]) # Or use your actual date/value here
print(my_list)

In [ ]:
# Create the figure and axes
fig, ax = plt.subplots(figsize=(16, 6))

# --- FIX 1: Use ax.plot() and DO NOT assign it back to 'ax' ---
# Line plot of predicted model
ax.plot(df_subset['datetime'], y_pred_model, 
        color='gray', linestyle='solid', linewidth=2, label='Predicted Model')

# Define consistent colors
water_periods = sorted(df_subset['WATER_PERIOD'].unique())
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
colors = {period: color_palette[i % len(color_palette)] for i, period in enumerate(water_periods)}

# --- FIX 2: Loop for points ---
for water_period in sorted(df_subset['WATER_PERIOD'].unique()):
    subset = df_subset[(df_subset['WATER_PERIOD'] == water_period)]
    
    # Use ax.plot (or ax.scatter) directly on the existing 'ax' object
    ax.plot(subset['datetime'], subset['TSS'],
             marker='o',
             markersize=6,
             alpha=0.5,
             color=colors[water_period],
             linestyle='none', # 'none' ensures points are not connected
             label=f'{water_period}')

# Get metrics for the text box
# (Ensure model_fit and cv_metrics are defined before running this)
r2 = round(float(cv_metrics['r2']), 4)
coef1 = round(float(model_fit.named_steps['linear'].coef_[0]), 4)
coef2 = round(float(model_fit.named_steps['linear'].coef_[1]), 4)
intercept = round(float(model_fit.named_steps['linear'].intercept_), 4)

r2_str = f'$R^2$ = {r2}'
equation = f"$y = {coef1}x + {coef2}x^2 + {intercept}$"

# Print to console (optional)
# print(equation)
# print(r2_str)

# --- FIX 3: ax.text works now because ax is still an Axes object ---
ax.text(0.05, 0.95, f'{equation}\n{r2_str}', 
        transform=ax.transAxes, fontsize=9,
        verticalalignment='top', 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

ax.set_xlabel('Year')
ax.set_ylabel('TSS (mg/L)')
ax.set_title('NIR Band vs TSS with Polynomial Regression Model')

# Move legend outside to the right
ax.legend(loc='upper right')

# Set X ticks
ax.set_xticks(ticks)
ax.set_xticklabels(my_list)

plt.tight_layout()
plt.show()

### Polynomial NIR + RED regression

In [ ]:
# separate parameters
y = df_subset['TSS'].copy()
X = df_subset[['nir_mean','red_mean']].copy()

In [ ]:
model_fit = model_poly.fit(X, y)

In [ ]:
print(f"coefficients: {model_fit.named_steps['linear'].coef_}")
print(f"intercept: {model_fit.named_steps['linear'].intercept_}")

In [ ]:
y_pred_model = model_fit.predict(X)

#### Avaliação do modelo

In [ ]:
metrics = model_metrics(y,y_pred_model)
print('Metrics:')
print(metrics)

Cross validation

In [ ]:
cv_metrics = cv_model_metrics(model=model_poly,X=X,y=y,n_cv=100)
print('CV Metrics:')
print(cv_metrics)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="actual_vs_predicted",
    ax=axs[0]
)
axs[0].set_title("Actual vs. Predicted values")
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="residual_vs_predicted",
    ax=axs[1]
)
axs[1].set_title("Residuals vs. Predicted Values")
fig.suptitle("Plotting predictions")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Create a list of 202 empty strings
total_len = len(y_pred_model)
print(total_len)
ticks = np.arange(0,total_len ,20)
print(ticks)
ticks_len = len(ticks)
my_list = []


for i in range(0, ticks_len):
    my_list.append(df_subset['date'].iloc[ticks[i]]) # Or use your actual date/value here
print(my_list)

In [ ]:
# Create the figure and axes
fig, ax = plt.subplots(figsize=(16, 6))

# --- FIX 1: Use ax.plot() and DO NOT assign it back to 'ax' ---
# Line plot of predicted model
ax.plot(df_subset['datetime'], y_pred_model, 
        color='gray', linestyle='solid', linewidth=2, label='Predicted Model')

# Define consistent colors
water_periods = sorted(df_subset['WATER_PERIOD'].unique())
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
colors = {period: color_palette[i % len(color_palette)] for i, period in enumerate(water_periods)}

# --- FIX 2: Loop for points ---
for water_period in sorted(df_subset['WATER_PERIOD'].unique()):
    subset = df_subset[(df_subset['WATER_PERIOD'] == water_period)]
    
    # Use ax.plot (or ax.scatter) directly on the existing 'ax' object
    ax.plot(subset['datetime'], subset['TSS'],
             marker='o',
             markersize=6,
             alpha=0.5,
             color=colors[water_period],
             linestyle='none', # 'none' ensures points are not connected
             label=f'{water_period}')

# Get metrics for the text box
# (Ensure model_fit and cv_metrics are defined before running this)
r2 = round(float(cv_metrics['r2']), 4)
coef1 = round(float(model_fit.named_steps['linear'].coef_[0]), 4)
coef2 = round(float(model_fit.named_steps['linear'].coef_[1]), 4)
intercept = round(float(model_fit.named_steps['linear'].intercept_), 4)

r2_str = f'$R^2$ = {r2}'
equation = f"$y = {coef1}x + {coef2}x^2 + {intercept}$"

# Print to console (optional)
# print(equation)
# print(r2_str)

# --- FIX 3: ax.text works now because ax is still an Axes object ---
ax.text(0.05, 0.95, f'{equation}\n{r2_str}', 
        transform=ax.transAxes, fontsize=9,
        verticalalignment='top', 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

ax.set_xlabel('Year')
ax.set_ylabel('TSS (mg/L)')
ax.set_title('NIR Band vs TSS with Polynomial Regression Model')

# Move legend outside to the right
ax.legend(loc='upper right')

# Set X ticks
ax.set_xticks(ticks)
ax.set_xticklabels(my_list)

plt.tight_layout()
plt.show()

## Multiple linear regression

In [ ]:
# separate parameters
y = df_subset['TSS'].copy()
X = df_subset.drop(drop_columns_X_model,axis = 1).copy()

In [ ]:
ols_fit = ols.fit(X, y)
y_pred_mlr = ols_fit.predict(X)

In [ ]:
print(f"intercept: {ols_fit.intercept_}")
for n in range(len(X.columns)):

    print(f"{X.columns[n]} coef: {ols_fit.coef_[n]}")

### Avaliação do modelo

In [ ]:
metrics = model_metrics(y,y_pred_mlr)
print('Metrics:')
print(metrics)

Cross validation

In [ ]:
cv_metrics = cv_model_metrics(model=ols,X=X,y=y)
print('CV Metrics:')
print(cv_metrics)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_mlr,
    kind="actual_vs_predicted",
    ax=axs[0]
)
axs[0].set_title("Actual vs. Predicted values")
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_mlr,
    kind="residual_vs_predicted",
    ax=axs[1]
)
axs[1].set_title("Residuals vs. Predicted Values")
fig.suptitle("Plotting predictions")
plt.tight_layout()
plt.show()

In [ ]:
sns.lineplot(data=df_subset, x ='datetime',y= y_pred_mlr,color='gray')
sns.pointplot(data=df_subset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')


### NIR / RED

In [ ]:
# separate parameters
y = df_subset['SPM'].copy()
X = df_subset['nir_mean']/ df_subset['red_mean']
X = np.array(X).reshape(-1, 1).copy()
X

In [ ]:
model_fit = model_poly.fit(X, y)
y_pred_mlr = model_fit.predict(X)

#### Avaliação do modelo

In [ ]:
metrics = model_metrics(y,y_pred_mlr)
print('Metrics:')
print(metrics)

Cross validation

In [ ]:
cv_metrics = cv_model_metrics(model=model_poly,X=X,y=y)
print('CV Metrics:')
print(cv_metrics)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_mlr,
    kind="actual_vs_predicted",
    ax=axs[0]
)
axs[0].set_title("Actual vs. Predicted values")
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_mlr,
    kind="residual_vs_predicted",
    ax=axs[1]
)
axs[1].set_title("Residuals vs. Predicted Values")
fig.suptitle("Plotting predictions")
plt.tight_layout()
plt.show()

In [ ]:
sns.lineplot(data=df_subset, x ='datetime',y= y_pred_mlr,color='gray')
sns.pointplot(data=df_subset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')


## Polynomial multiple regression

In [ ]:
# separate parameters
y = df_subset['SPM'].copy()
X = df_subset.drop(drop_columns_X_model,axis = 1).copy()

In [ ]:
model_fit = model_poly.fit(X, y)

In [ ]:
print(f"coefficients: {model_fit.named_steps['linear'].coef_}")
print(f"intercept: {model_fit.named_steps['linear'].intercept_}")

In [ ]:
y_pred_model = model_fit.predict(X)

### Avaliação do modelo

In [ ]:
metrics = model_metrics(y,y_pred_model)
print('Metrics:')
print(metrics)

Cross validation

In [ ]:
cv_metrics = cv_model_metrics(model=model_poly,X=X,y=y)
print('CV Metrics:')
print(cv_metrics)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="actual_vs_predicted",
    ax=axs[0]
)
axs[0].set_title("Actual vs. Predicted values")
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="residual_vs_predicted",
    ax=axs[1]
)
axs[1].set_title("Residuals vs. Predicted Values")
fig.suptitle("Plotting predictions")
plt.tight_layout()
plt.show()

In [ ]:
sns.lineplot(data=df_subset, x ='datetime',y= y_pred_model,color='gray')
sns.pointplot(data=df_subset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

## Splines (multiple)

In [ ]:
# separate parameters
y = df_subset['SPM'].copy()
X = df_subset.drop(drop_columns_X_model,axis = 1).copy()
# X = df_subset.drop('SPM',axis = 1).copy()

In [ ]:
model_fit = model_splines.fit(X, y)

In [ ]:
print(f"coefficients: {model_fit.named_steps['linear'].coef_}")
print(f"intercept: {model_fit.named_steps['linear'].intercept_}")

In [ ]:
y_pred_model = model_fit.predict(X)

### Avaliação do modelo

In [ ]:
metrics = model_metrics(y,y_pred_model)
print('Metrics:')
print(metrics)

Cross validation

In [ ]:
cv_metrics = cv_model_metrics(model=model_splines,X=X,y=y)
print('CV Metrics:')
print(cv_metrics)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="actual_vs_predicted",
    ax=axs[0]
)
axs[0].set_title("Actual vs. Predicted values")
PredictionErrorDisplay.from_predictions(
    y,
    y_pred=y_pred_model,
    kind="residual_vs_predicted",
    ax=axs[1]
)
axs[1].set_title("Residuals vs. Predicted Values")
fig.suptitle("Plotting predictions")
plt.tight_layout()
plt.show()

In [ ]:
sns.lineplot(data=df_subset, x ='datetime',y= y_pred_model,color='gray')
sns.pointplot(data=df_subset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

## Water Period

In [ ]:
# separate data per water period
water_periods = df_subset.WATER_PERIOD.unique()
df_periods_list = []
for period in water_periods:
    df = df_subset.loc[df_subset['WATER_PERIOD'] == period].copy()
    # print(df.head())
    df_periods_list.append(df)

### NIR and RED polynomial regression

#### 2nd degree

In [ ]:
metrics_list=[]
cv_metrics_list=[]
coef_list = []
intercept_list = []

for dataset in df_periods_list:
    # print(dataset['WATER_PERIOD'].iloc[0])
    y = dataset['SPM'].copy()

    X = dataset.loc[:,['nir_mean','red_mean']].copy()
    ols_fit = model_poly.fit(X, y)
    y_pred_mlr = ols_fit.predict(X)


    intercept_list.append(ols_fit.named_steps['linear'].intercept_)


    # print('equation')
    # print(f"intercept: {ols_fit.intercept_}")

    # for n in range(len(X.columns)):

    #     print(f"{X.columns[n]} coef: {ols_fit.coef_[n]}")

    coef_dic = dict(zip(X.columns, ols_fit.named_steps['linear'].coef_))

    coef_list.append(coef_dic)

    metrics = model_metrics(y,y_pred_mlr)
    metrics_list.append(metrics)

    cv_metrics = cv_model_metrics(model=ols,X=X,y=y)
    cv_metrics_list.append(cv_metrics)

    # print(' ')
    # print('Metrics:')
    # print(metrics)
    # print(' ')

    fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="actual_vs_predicted",
        ax=axs[0]
    )
    axs[0].set_title("Actual vs. Predicted values")
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="residual_vs_predicted",
        ax=axs[1]
    )
    axs[1].set_title("Residuals vs. Predicted Values")
    fig.suptitle("Plotting predictions")
    plt.tight_layout()
    plt.show()

    sns.lineplot(data=dataset, x ='datetime',y= y_pred_mlr,color='gray')
    sns.pointplot(data=dataset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

In [ ]:
dictionary_coef = dict(zip(water_periods, coef_list))
print(f"coefficients: {dictionary_coef}")
dictionary_intercep = dict(zip(water_periods, intercept_list))
print(f"intercepts: {dictionary_intercep}")
dictionary_metrics = dict(zip(water_periods, metrics_list))
print(f"metrics: {dictionary_metrics}")
dictionary_cv_metrics = dict(zip(water_periods, cv_metrics_list))
print(f"cv metrics: {dictionary_cv_metrics}")

### Multiple linear regession

In [ ]:
metrics_list=[]
coef_list = []
intercept_list = []
cv_metrics_list = []
for dataset in df_periods_list:
    # print(dataset['WATER_PERIOD'].iloc[0])
    y = dataset['SPM'].copy()

    X = dataset.drop(drop_columns_X_model,axis = 1).copy()
    ols_fit = ols.fit(X, y)
    y_pred_mlr = ols_fit.predict(X)


    intercept_list.append(ols_fit.intercept_)


    # print('equation')
    # print(f"intercept: {ols_fit.intercept_}")

    # for n in range(len(X.columns)):

    #     print(f"{X.columns[n]} coef: {ols_fit.coef_[n]}")

    coef_dic = dict(zip(X.columns, ols_fit.coef_))

    coef_list.append(coef_dic)

    metrics = model_metrics(y,y_pred_mlr)
    metrics_list.append(metrics)

    # cv_metrics = cv_model_metrics(model=ols,X=X,y=y)
    # cv_metrics_list.append(cv_metrics)
    # print(' ')
    # print('Metrics:')
    # print(metrics)
    # print(' ')

    fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="actual_vs_predicted",
        ax=axs[0]
    )
    axs[0].set_title("Actual vs. Predicted values")
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="residual_vs_predicted",
        ax=axs[1]
    )
    axs[1].set_title("Residuals vs. Predicted Values")
    fig.suptitle("Plotting predictions")
    plt.tight_layout()
    plt.show()

    sns.lineplot(data=dataset, x ='datetime',y= y_pred_mlr,color='gray')
    sns.pointplot(data=dataset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

In [ ]:
dictionary_coef = dict(zip(water_periods, coef_list))
print(f"coefficients: {dictionary_coef}")
dictionary_intercep = dict(zip(water_periods, intercept_list))
print(f"intercepts: {dictionary_intercep}")
dictionary_metrics = dict(zip(water_periods, metrics_list))
print(f"metrics: {dictionary_metrics}")
# dictionary_cv_metrics = dict(zip(water_periods, cv_metrics_list))
# print(f"cv metrics: {dictionary_cv_metrics}")

### Multiple polynomial regession

#### 2nd degree

In [ ]:
metrics_list=[]
cv_metrics_list=[]
coef_list = []
intercept_list = []

for dataset in df_periods_list:
    # print(dataset['WATER_PERIOD'].iloc[0])
    y = dataset['SPM'].copy()

    X = dataset.drop(drop_columns_X_model,axis = 1).copy()
    ols_fit = model_poly.fit(X, y)
    y_pred_mlr = ols_fit.predict(X)


    intercept_list.append(ols_fit.named_steps['linear'].intercept_)


    # print('equation')
    # print(f"intercept: {ols_fit.intercept_}")

    # for n in range(len(X.columns)):

    #     print(f"{X.columns[n]} coef: {ols_fit.coef_[n]}")

    coef_dic = dict(zip(X.columns, ols_fit.named_steps['linear'].coef_))

    coef_list.append(coef_dic)

    metrics = model_metrics(y,y_pred_mlr)
    metrics_list.append(metrics)

    cv_metrics = cv_model_metrics(model=ols,X=X,y=y)
    cv_metrics_list.append(cv_metrics)

    # print(' ')
    # print('Metrics:')
    # print(metrics)
    # print(' ')

    fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="actual_vs_predicted",
        ax=axs[0]
    )
    axs[0].set_title("Actual vs. Predicted values")
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="residual_vs_predicted",
        ax=axs[1]
    )
    axs[1].set_title("Residuals vs. Predicted Values")
    fig.suptitle("Plotting predictions")
    plt.tight_layout()
    plt.show()

    sns.lineplot(data=dataset, x ='datetime',y= y_pred_mlr,color='gray')
    sns.pointplot(data=dataset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

In [ ]:
dictionary_coef = dict(zip(water_periods, coef_list))
print(f"coefficients: {dictionary_coef}")
dictionary_intercep = dict(zip(water_periods, intercept_list))
print(f"intercepts: {dictionary_intercep}")
dictionary_metrics = dict(zip(water_periods, metrics_list))
print(f"metrics: {dictionary_metrics}")
dictionary_cv_metrics = dict(zip(water_periods, cv_metrics_list))
print(f"cv metrics: {dictionary_cv_metrics}")

#### 3rd degree

In [ ]:
model_poly = Pipeline([('poly', PolynomialFeatures(degree=3, include_bias=False)),
                  ('linear', LinearRegression(positive=False))])

In [ ]:
metrics_list=[]
coef_list = []
intercept_list = []
cv_metrics_list=[]

for dataset in df_periods_list:
    # print(dataset['WATER_PERIOD'].iloc[0])
    y = dataset['SPM'].copy()

    X = dataset.drop(drop_columns_X_model,axis = 1).copy()
    ols_fit = model_poly.fit(X, y)
    y_pred_mlr = ols_fit.predict(X)


    intercept_list.append(ols_fit.named_steps['linear'].intercept_)


    # print('equation')
    # print(f"intercept: {ols_fit.intercept_}")

    # for n in range(len(X.columns)):

    #     print(f"{X.columns[n]} coef: {ols_fit.coef_[n]}")

    coef_dic = dict(zip(X.columns, ols_fit.named_steps['linear'].coef_))

    coef_list.append(coef_dic)

    metrics = model_metrics(y,y_pred_mlr)
    metrics_list.append(metrics)

    cv_metrics = cv_model_metrics(model=model_poly,X=X,y=y)
    cv_metrics_list.append(cv_metrics)

    # print(' ')
    # print('Metrics:')
    # print(metrics)
    # print(' ')

    fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="actual_vs_predicted",
        ax=axs[0]
    )
    axs[0].set_title("Actual vs. Predicted values")
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="residual_vs_predicted",
        ax=axs[1]
    )
    axs[1].set_title("Residuals vs. Predicted Values")
    fig.suptitle("Plotting predictions")
    plt.tight_layout()
    plt.show()

    sns.lineplot(data=dataset, x ='datetime',y= y_pred_mlr,color='gray')
    sns.pointplot(data=dataset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

In [ ]:
dictionary_coef = dict(zip(water_periods, coef_list))
print(f"coefficients: {dictionary_coef}")
dictionary_intercep = dict(zip(water_periods, intercept_list))
print(f"intercepts: {dictionary_intercep}")
dictionary_metrics = dict(zip(water_periods, metrics_list))
print(f"metrics: {dictionary_metrics}")

dictionary_cv_metrics = dict(zip(water_periods, cv_metrics_list))
print(f"cv metrics: {dictionary_cv_metrics}")

## multiple spline regression

In [ ]:
metrics_list=[]
cv_metrics_list=[]
coef_list = []
intercept_list = []

for dataset in df_periods_list:
    # print(dataset['WATER_PERIOD'].iloc[0])
    y = dataset['SPM'].copy()

    X = dataset.drop(drop_columns_X_model,axis = 1).copy()
    ols_fit = model_splines.fit(X, y)
    y_pred_mlr = ols_fit.predict(X)


    intercept_list.append(ols_fit.named_steps['linear'].intercept_)


    # print('equation')
    # print(f"intercept: {ols_fit.intercept_}")

    # for n in range(len(X.columns)):

    #     print(f"{X.columns[n]} coef: {ols_fit.coef_[n]}")

    coef_dic = dict(zip(X.columns, ols_fit.named_steps['linear'].coef_))

    coef_list.append(coef_dic)

    metrics = model_metrics(y,y_pred_mlr)
    metrics_list.append(metrics)

    cv_metrics = cv_model_metrics(model =model_splines,X=X,y=y )
    cv_metrics_list.append(cv_metrics)

    # print(' ')
    # print('Metrics:')
    # print(metrics)
    # print(' ')

    fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="actual_vs_predicted",
        ax=axs[0]
    )
    axs[0].set_title("Actual vs. Predicted values")
    PredictionErrorDisplay.from_predictions(
        y,
        y_pred=y_pred_mlr,
        kind="residual_vs_predicted",
        ax=axs[1]
    )
    axs[1].set_title("Residuals vs. Predicted Values")
    fig.suptitle("Plotting predictions")
    plt.tight_layout()
    plt.show()

    sns.lineplot(data=dataset, x ='datetime',y= y_pred_mlr,color='gray')
    sns.pointplot(data=dataset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

In [ ]:
dictionary_coef = dict(zip(water_periods, coef_list))
print(f"coefficients: {dictionary_coef}")
dictionary_intercep = dict(zip(water_periods, intercept_list))
print(f"intercepts: {dictionary_intercep}")
dictionary_metrics = dict(zip(water_periods, metrics_list))
print(f"metrics: {dictionary_metrics}")

dictionary_cv_metrics = dict(zip(water_periods, cv_metrics_list))
print(f"cv metrics: {dictionary_cv_metrics}")

## Bands & Water Period

In [ ]:
metrics_list=[]
cv_metrics_list=[]
coef_list = []
intercept_list = []
bands = ['blue_mean','green_mean','red_mean','nir_mean']
for dataset in df_periods_list:
    # print(dataset['WATER_PERIOD'].iloc[0])
    y = dataset['SPM'].copy()

    for band in bands:

        X = np.array(dataset[band]).reshape(-1, 1).copy()
        ols_fit = ols.fit(X, y)
        y_pred_mlr = ols_fit.predict(X)


        intercept_list.append(ols_fit.intercept_)


        # print('equation')
        # print(f"intercept: {ols_fit.intercept_}")

        # for n in range(len(X.columns)):

        #     print(f"{X.columns[n]} coef: {ols_fit.coef_[n]}")

        coef_dic = {band:ols_fit.coef_}

        coef_list.append(coef_dic)

        metrics = model_metrics(y,y_pred_mlr)
        metrics_list.append(metrics)

        cv_metrics = cv_model_metrics(model = ols,X=X,y=y)
        cv_metrics_list.append(cv_metrics)
        # print(' ')
        # print('Metrics:')
        # print(metrics)
        # print(' ')

        fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
        PredictionErrorDisplay.from_predictions(
            y,
            y_pred=y_pred_mlr,
            kind="actual_vs_predicted",
            ax=axs[0]
        )
        axs[0].set_title("Actual vs. Predicted values")
        PredictionErrorDisplay.from_predictions(
            y,
            y_pred=y_pred_mlr,
            kind="residual_vs_predicted",
            ax=axs[1]
        )
        axs[1].set_title("Residuals vs. Predicted Values")
        fig.suptitle(f"Plotting predictions: {band}")
        plt.tight_layout()
        plt.show()

        sns.lineplot(data=dataset, x ='datetime',y= y_pred_mlr,color='gray')
        sns.pointplot(data=dataset, x ='datetime',y= y, linestyle="none", markersize=5, alpha=.6,hue='WATER_PERIOD')

In [ ]:
metrics_list
cv_metrics_list
coef_list
intercept_list
bands
water_periods

In [ ]:
dic_list = []
for i in range(len(water_periods)):
    for j in range(len(bands)):
        dic_list.append(dict(band = bands[j],
                   period = water_periods[i],
                   coefficients = coef_list[(i+1)*(j)],
                   intercept = intercept_list[(i+1)*(j)],
                   metrics = metrics_list[(i+1)*(j)],
                   cv_metrics = cv_metrics_list[(i+1)*(j)]))


In [ ]:
print(dic_list)